# 🎬 GURU KULA DESAM — BULK DEVOTIONAL STUDIO
### Autonomous Pipeline: Copy All 317 Audio Tracks ➔ Generate Thematic 24-Image Suites ➔ Assemble 720p 3D Arc Master Films

In [ ]:
# Cell 1: Environment Setup & High-Speed Diffusion Engine
!nvidia-smi
!pip install -q diffusers transformers accelerate safetensors Pillow mutagen

import torch
from diffusers import StableDiffusionXLPipeline, UNet2DConditionModel, EulerDiscreteScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

print("⏳ Loading SDXL-Lightning into GPU VRAM...")
base = "stabilityai/stable-diffusion-xl-base-1.0"
repo = "ByteDance/SDXL-Lightning"
ckpt = "sdxl_lightning_4step_unet.safetensors"

unet = UNet2DConditionModel.from_config(base, subfolder="unet").to("cuda", torch.float16)
unet.load_state_dict(load_file(hf_hub_download(repo, ckpt), device="cuda"))
pipe = StableDiffusionXLPipeline.from_pretrained(base, unet=unet, torch_dtype=torch.float16, variant="fp16").to("cuda")
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")
print("✅ SDXL-Lightning loaded on GPU (Zero API key / Zero external credits required)!")


In [ ]:
# Cell 2: Fast Copy of ALL 317 Audio Tracks from Google Drive into Local Colab /content/Music
import os, shutil, glob
from google.colab import drive

drive.mount("/content/drive")

source_dir = "/content/drive/MyDrive/PC_Backup_GuruKulaDesam/Downloads"
dest_dir = "/content/Music"
os.makedirs(dest_dir, exist_ok=True)

print(f"⏳ Copying all WAV files from {source_dir} to {dest_dir}...")
copied = 0
for f in os.listdir(source_dir):
    if f.lower().endswith(".wav"):
        src = os.path.join(source_dir, f)
        dst = os.path.join(dest_dir, f)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
        copied += 1

print(f"✅ Successfully copied {copied} WAV audio tracks locally to {dest_dir}!")


In [ ]:
# Cell 3: Bulk Generate All 24 Thematic Scenes per Song Across All Tracks
import os, json, unicodedata
from pathlib import Path
from PIL import Image

# Clone latest repository for scripts and overlays
if not os.path.exists("/content/GuruKulaDesam"):
    !git clone -b main https://github.com/shishyan/GuruKulaDesam.git /content/GuruKulaDesam

with open("/content/GuruKulaDesam/tools/all_devotional_prompts.json", "r", encoding="utf-8") as f:
    PROMPT_REGISTRY = json.load(f)

STYLE_PREFIX = (
    "A 16:9 cinematic photorealistic devotional photograph, 8k, Tamil Nadu, vivid, warm, and sacred. "
    "Temples look majestic after a sacred maha kumbabishekam: intricately carved gopuram, glowing gold kalasams, "
    "polished granite sanctum floors reflecting flickering brass oil lamps, fragrant marigold and jasmine garlands, "
    "sacred dhoopam incense smoke wafting. Devotees in clean traditional festive silk attire. 16:9 composition: "
)

print(f"Loaded {len(PROMPT_REGISTRY)} theme suites with {sum(len(p) for p in PROMPT_REGISTRY.values())} total prompts.")

for theme_key, prompts in PROMPT_REGISTRY.items():
    out_dir = Path(f"/content/GuruKulaDesam/production/scene_images/{theme_key}")
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"
🎨 Generating 24 scenes for: {theme_key}...")
    for idx, p in enumerate(prompts):
        img_path = out_dir / f"scene_{idx+1:02d}.jpg"
        if img_path.exists() and img_path.stat().st_size > 10000:
            continue
        prompt_text = STYLE_PREFIX + p
        image = pipe(prompt=prompt_text, num_inference_steps=4, guidance_scale=0.0, width=1280, height=720).images[0]
        image.save(img_path, quality=95)
        print(f"  ✔ {theme_key} scene_{idx+1:02d}.jpg saved", flush=True)

print("
🎉 ALL THEMATIC SCENE IMAGES HAVE BEEN GENERATED AND ARE READY FOR ASSEMBLY!")


In [ ]:
# Cell 4: Bulk Video Assembly (3D Arc Camera Moves + Reverse Pendulum + NVENC)
import os, glob, subprocess, shutil, unicodedata
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

THEME_AUDIO_MAP = {
    "thirueragam": "திருஏரகம்",
    "kunruthoradal": "குன்றுதோறாடல்",
    "pazhamudircholai": "பழமுதிர் சோலை",
    "thiruchendur": "திருச்செந்தூர் 1Q",
    "thodu_udaiya_seviyan": "தோடு உடைய செவியன்",
    "patharodu": "பத்தரோடு",
    "maim_maru": "மைம் மரு",
    "sei_aruge": "செய் அருகே",
    "angamum_vedhamum": "அங்கமும் வேதமும் 1N Golden 2026",
    "paadaga_melladi": "பாடக மெல்லடி",
    "punniyar_poothiyar": "புண்ணியர் பூதியர்",
}

music_dir = "/content/Music"
renders_dir = "/content/GuruKulaDesam/renders/releases"
os.makedirs(renders_dir, exist_ok=True)

def assemble_theme_film(theme, search_keyword):
    img_dir = Path(f"/content/GuruKulaDesam/production/scene_images/{theme}")
    imgs = sorted(list(img_dir.glob("*.jpg")))
    if len(imgs) < 12:
        print(f"⚠️ Skipping {theme}: only {len(imgs)} images found.")
        return

    # Match audio in /content/Music
    matched_audio = None
    for fname in os.listdir(music_dir):
        norm = unicodedata.normalize("NFC", fname)
        if search_keyword in norm and norm.endswith(".wav"):
            matched_audio = os.path.join(music_dir, fname)
            break

    if not matched_audio:
        print(f"⚠️ Audio for {theme} ({search_keyword}) not found in {music_dir}.")
        return

    out_film = f"{renders_dir}/{theme}_cinematic_720p_master.mp4"
    if os.path.exists(out_film):
        print(f"⏩ {theme} already rendered: {out_film}")
        return

    print(f"
==================================================")
    print(f"🎬 ASSEMBLING: {theme} ({Path(matched_audio).name})")
    print(f"==================================================")
    
    adur = float(subprocess.check_output(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", matched_audio]).decode().strip())
    FPS, SHOT_DUR, FADE_DUR = 24, 6.5, 1.0
    effective = SHOT_DUR - FADE_DUR
    needed = int((adur + 1.0) / effective) + 1
    
    # Pendulum reverse bounce
    cycle = 2 * (len(imgs) - 1)
    selected = [imgs[i % cycle if (i % cycle) < len(imgs) else cycle - (i % cycle)] for i in range(needed)]
    
    tmp = Path(f"/content/temp_{theme}")
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True, exist_ok=True)
    
    def render_clip(item):
        i, im = item
        out_clip = tmp / f"clip_{i:03d}.mp4"
        m = i % 6
        if m == 0:
            vf = f"scale=1536:864:flags=bicubic,crop=1280:720:(in_w-out_w)*(0.5+0.48*cos(PI*t/{SHOT_DUR})):(in_h-out_h)*(0.5+0.45*sin(PI*t/{SHOT_DUR})),fps={FPS},setsar=1"
        elif m == 1:
            vf = f"scale=1536:864:flags=bicubic,crop=1280:720:(in_w-out_w)*(0.5-0.48*cos(PI*t/{SHOT_DUR})):(in_h-out_h)*(0.5-0.45*sin(PI*t/{SHOT_DUR})),fps={FPS},setsar=1"
        elif m == 2:
            vf = f"scale=1536:864:flags=bicubic,crop=1280:720:(in_w-out_w)*t/{SHOT_DUR}:(in_h-out_h)*(1-t/{SHOT_DUR}),fps={FPS},setsar=1"
        elif m == 3:
            vf = f"scale=1536:864:flags=bicubic,crop=1280:720:(in_w-out_w)*(1-t/{SHOT_DUR}):(in_h-out_h)*t/{SHOT_DUR},fps={FPS},setsar=1"
        elif m == 4:
            vf = f"scale=1536:864:flags=bicubic,crop=1280:720:(in_w-out_w)*t/{SHOT_DUR}:(in_h-out_h)/2,fps={FPS},setsar=1"
        else:
            vf = f"scale=1536:864:flags=bicubic,crop=1280:720:(in_w-out_w)*(1-t/{SHOT_DUR}):(in_h-out_h)/2,fps={FPS},setsar=1"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-loop", "1", "-t", str(SHOT_DUR), "-i", str(im), "-vf", vf, "-c:v", "h264_nvenc", "-preset", "p4", "-cq", "20", "-pix_fmt", "yuv420p", "-an", str(out_clip)], check=True)
        return i, out_clip
    
    clips = [None] * len(selected)
    with ThreadPoolExecutor(max_workers=16) as ex:
        for idx, cp in ex.map(render_clip, enumerate(selected)):
            clips[idx] = cp
    
    chunk_size = 8
    chunks = [clips[c:c + chunk_size] for c in range(0, len(clips), chunk_size)]
    
    def render_chunk(item):
        s_idx, ch = item
        if len(ch) == 1: return s_idx, ch[0]
        s_out = tmp / f"stage_{s_idx:02d}.mp4"
        inputs = [arg for c in ch for arg in ("-i", str(c))]
        fc, prev, off = [], "[0:v]", SHOT_DUR - FADE_DUR
        for j in range(1, len(ch)):
            nxt = f"[{j}:v]"
            out_lbl = f"[v{j}]" if j < len(ch) - 1 else "[outv]"
            fc.append(f"{prev}{nxt}xfade=transition=fade:duration={FADE_DUR}:offset={off:.2f}{out_lbl}")
            prev, off = out_lbl, off + (SHOT_DUR - FADE_DUR)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", *inputs, "-filter_complex", ";".join(fc), "-map", "[outv]", "-c:v", "h264_nvenc", "-preset", "p4", "-cq", "20", "-pix_fmt", "yuv420p", str(s_out)], check=True)
        return s_idx, s_out
    
    stages = [None] * len(chunks)
    with ThreadPoolExecutor(max_workers=min(8, len(chunks))) as ex:
        for s_idx, sp in ex.map(render_chunk, enumerate(chunks)): stages[s_idx] = sp
    
    base_video = stages[0] if len(stages) == 1 else tmp / "base_video.mp4"
    if len(stages) > 1:
        inputs = [arg for s in stages for arg in ("-i", str(s))]
        fc, prev = [], "[0:v]"
        off = float(subprocess.check_output(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", str(stages[0])]).decode().strip()) - FADE_DUR
        for j in range(1, len(stages)):
            nxt = f"[{j}:v]"
            out_lbl = f"[sv{j}]" if j < len(stages) - 1 else "[basev]"
            fc.append(f"{prev}{nxt}xfade=transition=fade:duration={FADE_DUR}:offset={off:.2f}{out_lbl}")
            prev = out_lbl
            off += (float(subprocess.check_output(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", str(stages[j])]).decode().strip()) - FADE_DUR)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", *inputs, "-filter_complex", ";".join(fc), "-map", "[basev]", "-c:v", "h264_nvenc", "-preset", "p4", "-cq", "20", "-pix_fmt", "yuv420p", str(base_video)], check=True)
    
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", str(base_video), "-i", str(matched_audio),
        "-c:v", "copy", "-c:a", "aac", "-b:a", "256k",
        "-af", f"afade=t=out:st={max(0, adur-2):.2f}:d=2",
        "-t", f"{adur:.2f}", str(out_film)
    ], check=True)
    print(f"✔ COMPLETED: {out_film}
")

for theme, search_kw in THEME_AUDIO_MAP.items():
    assemble_theme_film(theme, search_kw)
print("
🎉 BULK CINEMATIC MASTER FILM ASSEMBLY FINISHED!")
